# Comparison of Effective Sample Size Across Setups A-E

This notebook compares the effective sample size (ESS) across Setups A-E under one aligned operating basis.

The main output is one comparison table covering:
- Setup A at 5 mm probe diameter and 1.8 s acquisition time
- Setup B at 5 mm probe diameter, 95 kg/h throughput metadata, 3.6 s acquisition time, and 3 s spoon interval
- Setup C at 5 mm probe diameter, 95 kg/h throughput, 1.8 s acquisition time, and 32 RPM
- Setup D at 5 mm probe diameter, 95 kg/h throughput, and 1.8 s acquisition time
- Setup E at 95 kg/h throughput, 1.8 s acquisition time, and probe diameters of 5 mm and 25 mm

The 1.8 s spoon scenario for Setup B is intentionally omitted.

## Shared Assumptions

- Bulk density is fixed at 0.4 g/cm$^3$ for every comparison row.
- Penetration depth is aligned to 1.0 mm for every setup because a common optical depth is required for direct comparison and 1.0 mm is already used as a representative row in the existing setup reports.
- Setup A is static, so acquisition time is reported for alignment only and does not change the ESS value.
- Setup B keeps 95 kg/h as aligned metadata, but the existing deterministic spoon model does not use throughput directly.
- Setup E uses the swept-capsule unique-exposure ESS proxy from the PAT-rig notebook.

The comparison table is the primary notebook output.

In [1]:
from __future__ import annotations

import importlib
import math
from pathlib import Path

import pandas as pd

pd.options.display.float_format = lambda value: f"{value:,.3f}"


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "reports").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "compare_effective_sample_size_across_setups.ipynb"
REPORTS_ROOT = PROJECT_ROOT / "reports"

ALIGNED_PENETRATION_DEPTH_MM = 1.0
ALIGNED_DENSITY_G_CM3 = 0.4
ALIGNED_THROUGHPUT_KG_H = 95.0
ALIGNED_ACQUISITION_TIME_S = 1.8

PROJECT_ROOT

WindowsPath('C:/temp files/EFS')

In [2]:
comparison_scenarios = [
    {"Setup": "A", "Scenario": "A1", "Basis": "Static illuminated volume", "Probe mm": 5.0, "Depth mm": ALIGNED_PENETRATION_DEPTH_MM, "Throughput kg/h": math.nan, "Acq s": 1.8, "Spoon s": math.nan, "RPM": math.nan, "Density g/cm^3": ALIGNED_DENSITY_G_CM3},
    {"Setup": "B", "Scenario": "B2", "Basis": "Static mass times spoon presentations", "Probe mm": 5.0, "Depth mm": ALIGNED_PENETRATION_DEPTH_MM, "Throughput kg/h": ALIGNED_THROUGHPUT_KG_H, "Acq s": 3.6, "Spoon s": 3.0, "RPM": math.nan, "Density g/cm^3": ALIGNED_DENSITY_G_CM3},
    {"Setup": "C", "Scenario": "C1", "Basis": "Blocked deterministic renewal", "Probe mm": 5.0, "Depth mm": ALIGNED_PENETRATION_DEPTH_MM, "Throughput kg/h": ALIGNED_THROUGHPUT_KG_H, "Acq s": 1.8, "Spoon s": math.nan, "RPM": 32.0, "Density g/cm^3": ALIGNED_DENSITY_G_CM3},
    {"Setup": "D", "Scenario": "D1", "Basis": "Flow-refresh proxy", "Probe mm": 5.0, "Depth mm": ALIGNED_PENETRATION_DEPTH_MM, "Throughput kg/h": ALIGNED_THROUGHPUT_KG_H, "Acq s": 1.8, "Spoon s": math.nan, "RPM": math.nan, "Density g/cm^3": ALIGNED_DENSITY_G_CM3},
    {"Setup": "E", "Scenario": "E1", "Basis": "Swept capsule unique exposure", "Probe mm": 5.0, "Depth mm": ALIGNED_PENETRATION_DEPTH_MM, "Throughput kg/h": ALIGNED_THROUGHPUT_KG_H, "Acq s": 1.8, "Spoon s": math.nan, "RPM": math.nan, "Density g/cm^3": ALIGNED_DENSITY_G_CM3},
    {"Setup": "E", "Scenario": "E2", "Basis": "Swept capsule unique exposure", "Probe mm": 25.0, "Depth mm": ALIGNED_PENETRATION_DEPTH_MM, "Throughput kg/h": ALIGNED_THROUGHPUT_KG_H, "Acq s": 1.8, "Spoon s": math.nan, "RPM": math.nan, "Density g/cm^3": ALIGNED_DENSITY_G_CM3},
]

scenario_df = pd.DataFrame(comparison_scenarios)
scenario_df

,Setup,Scenario,Basis,Probe mm,Depth mm,Throughput kg/h,Acq s,Spoon s,RPM,Density g/cm^3
0,A,A1,Static illuminated volume,5.000,1.000,NaN,1.800,NaN,NaN,0.400
1,B,B2,Static mass times spoon presentations,5.000,1.000,95.000,3.600,3.000,NaN,0.400
2,C,C1,Blocked deterministic renewal,5.000,1.000,95.000,1.800,NaN,32.000,0.400
3,D,D1,Flow-refresh proxy,5.000,1.000,95.000,1.800,NaN,NaN,0.400
4,E,E1,Swept capsule unique exposure,5.000,1.000,95.000,1.800,NaN,NaN,0.400
5,E,E2,Swept capsule unique exposure,25.000,1.000,95.000,1.800,NaN,NaN,0.400


## Calculation Basis

Setups A, B, and E are calculated directly from their notebook models with the aligned parameters above. Setups C and D are pulled from the latest exported report tables at the exact aligned operating point, so the comparison notebook reuses the established implementation instead of recreating the full mechanistic code path.

In [3]:
def latest_run_dir(setup_name: str) -> Path:
    setup_dir = REPORTS_ROOT / setup_name
    run_dirs = sorted(path for path in setup_dir.iterdir() if path.is_dir())
    if not run_dirs:
        raise FileNotFoundError(f"No report runs found for {setup_name}.")
    return run_dirs[-1]


def load_results_full(setup_name: str) -> pd.DataFrame:
    return pd.read_csv(latest_run_dir(setup_name) / "results_full.csv")


def circle_area_cm2(probe_diameter_mm: float) -> float:
    diameter_cm = probe_diameter_mm / 10.0
    return math.pi * (diameter_cm / 2.0) ** 2


def static_sampled_mass_mg(probe_diameter_mm: float, penetration_depth_mm: float, density_g_cm3: float) -> float:
    area_cm2 = circle_area_cm2(probe_diameter_mm)
    depth_cm = penetration_depth_mm / 10.0
    volume_cm3 = area_cm2 * depth_cm
    return density_g_cm3 * volume_cm3 * 1000.0


def calculate_setup_b_ess_mg(
    probe_diameter_mm: float,
    penetration_depth_mm: float,
    density_g_cm3: float,
    acquisition_time_s: float,
    spoon_interval_s: float,
) -> tuple[float, float, float]:
    static_mass_mg = static_sampled_mass_mg(probe_diameter_mm, penetration_depth_mm, density_g_cm3)
    spoons_contributing = acquisition_time_s / spoon_interval_s
    effective_mass_mg = static_mass_mg * spoons_contributing
    return static_mass_mg, spoons_contributing, effective_mass_mg


def calculate_setup_e_unique_exposed_ess(
    probe_diameter_mm: float,
    penetration_depth_mm: float,
    density_g_cm3: float,
    throughput_kg_h: float,
    acquisition_time_s: float,
    pipe_diameter_cm: float = 10.0,
) -> dict[str, float]:
    spot_diameter_cm = probe_diameter_mm / 10.0
    spot_area_cm2 = circle_area_cm2(probe_diameter_mm)
    depth_cm = penetration_depth_mm / 10.0
    mass_flow_g_s = throughput_kg_h * 1000.0 / 3600.0
    vol_flow_cm3_s = mass_flow_g_s / density_g_cm3
    pipe_area_cm2 = math.pi * (pipe_diameter_cm / 2.0) ** 2
    velocity_cm_s = vol_flow_cm3_s / pipe_area_cm2
    advance_cm = velocity_cm_s * acquisition_time_s
    swept_increment_area_cm2 = spot_diameter_cm * advance_cm
    probe_mass_mg_instant = density_g_cm3 * spot_area_cm2 * depth_cm * 1000.0
    swept_increment_mass_mg = density_g_cm3 * swept_increment_area_cm2 * depth_cm * 1000.0
    effective_sample_size_mg = probe_mass_mg_instant + swept_increment_mass_mg
    return {
        "velocity_cm_s": velocity_cm_s,
        "advance_mm_per_acq": advance_cm * 10.0,
        "probe_mass_mg_instant": probe_mass_mg_instant,
        "swept_increment_mass_mg": swept_increment_mass_mg,
        "effective_sample_size_mg": effective_sample_size_mg,
        "mass_passed_mg_per_acq": mass_flow_g_s * acquisition_time_s * 1000.0,
    }

In [4]:
setup_c_results = load_results_full("setup_c")
setup_d_results = load_results_full("setup_d")

setup_c_match = (
    (setup_c_results["density_g_cm3"] == ALIGNED_DENSITY_G_CM3)
    & (setup_c_results["rpm"] == 32)
    & (setup_c_results["penetration_depth_mm"] == ALIGNED_PENETRATION_DEPTH_MM)
    & (setup_c_results["probe_diameter_mm"] == 5.0)
    & (setup_c_results["acquisition_time_s"] == ALIGNED_ACQUISITION_TIME_S)
)
if setup_c_match.sum() != 1:
    raise ValueError("Expected exactly one Setup C comparison row.")
setup_c_row = setup_c_results.loc[setup_c_match].iloc[0]

setup_d_match = (
    (setup_d_results["throughput_kg_h"] == ALIGNED_THROUGHPUT_KG_H)
    & (setup_d_results["density_g_cm3"] == ALIGNED_DENSITY_G_CM3)
    & (setup_d_results["penetration_depth_mm"] == ALIGNED_PENETRATION_DEPTH_MM)
    & (setup_d_results["probe_diameter_mm"] == 5.0)
    & (setup_d_results["acquisition_time_s"] == ALIGNED_ACQUISITION_TIME_S)
)
if setup_d_match.sum() != 1:
    raise ValueError("Expected exactly one Setup D comparison row.")
setup_d_row = setup_d_results.loc[setup_d_match].iloc[0]

setup_a_ess_mg = static_sampled_mass_mg(5.0, ALIGNED_PENETRATION_DEPTH_MM, ALIGNED_DENSITY_G_CM3)
setup_b_static_mass_mg, setup_b_spoons_contributing, setup_b_ess_mg = calculate_setup_b_ess_mg(
    probe_diameter_mm=5.0,
    penetration_depth_mm=ALIGNED_PENETRATION_DEPTH_MM,
    density_g_cm3=ALIGNED_DENSITY_G_CM3,
    acquisition_time_s=3.6,
    spoon_interval_s=3.0,
)
setup_e_5mm = calculate_setup_e_unique_exposed_ess(
    probe_diameter_mm=5.0,
    penetration_depth_mm=ALIGNED_PENETRATION_DEPTH_MM,
    density_g_cm3=ALIGNED_DENSITY_G_CM3,
    throughput_kg_h=ALIGNED_THROUGHPUT_KG_H,
    acquisition_time_s=ALIGNED_ACQUISITION_TIME_S,
)
setup_e_25mm = calculate_setup_e_unique_exposed_ess(
    probe_diameter_mm=25.0,
    penetration_depth_mm=ALIGNED_PENETRATION_DEPTH_MM,
    density_g_cm3=ALIGNED_DENSITY_G_CM3,
    throughput_kg_h=ALIGNED_THROUGHPUT_KG_H,
    acquisition_time_s=ALIGNED_ACQUISITION_TIME_S,
)

comparison_rows = [
    {
        "setup": "A",
        "scenario": "A1",
        "basis": "Static illuminated volume",
        "probe_diameter_mm": 5.0,
        "penetration_depth_mm": ALIGNED_PENETRATION_DEPTH_MM,
        "throughput_kg_h": math.nan,
        "acquisition_time_s": 1.8,
        "spoon_interval_s": math.nan,
        "wheel_speed_rpm": math.nan,
        "density_g_cm3": ALIGNED_DENSITY_G_CM3,
        "ess_mg": setup_a_ess_mg,
    },
    {
        "setup": "B",
        "scenario": "B2",
        "basis": "Static mass times spoon presentations",
        "probe_diameter_mm": 5.0,
        "penetration_depth_mm": ALIGNED_PENETRATION_DEPTH_MM,
        "throughput_kg_h": ALIGNED_THROUGHPUT_KG_H,
        "acquisition_time_s": 3.6,
        "spoon_interval_s": 3.0,
        "wheel_speed_rpm": math.nan,
        "density_g_cm3": ALIGNED_DENSITY_G_CM3,
        "ess_mg": setup_b_ess_mg,
    },
    {
        "setup": "C",
        "scenario": "C1",
        "basis": "Blocked deterministic renewal",
        "probe_diameter_mm": float(setup_c_row["probe_diameter_mm"]),
        "penetration_depth_mm": float(setup_c_row["penetration_depth_mm"]),
        "throughput_kg_h": ALIGNED_THROUGHPUT_KG_H,
        "acquisition_time_s": float(setup_c_row["acquisition_time_s"]),
        "spoon_interval_s": math.nan,
        "wheel_speed_rpm": float(setup_c_row["rpm"]),
        "density_g_cm3": float(setup_c_row["density_g_cm3"]),
        "ess_mg": float(setup_c_row["effective_sampled_mass_estimate_mg"]),
    },
    {
        "setup": "D",
        "scenario": "D1",
        "basis": "Flow-refresh proxy",
        "probe_diameter_mm": float(setup_d_row["probe_diameter_mm"]),
        "penetration_depth_mm": float(setup_d_row["penetration_depth_mm"]),
        "throughput_kg_h": float(setup_d_row["throughput_kg_h"]),
        "acquisition_time_s": float(setup_d_row["acquisition_time_s"]),
        "spoon_interval_s": math.nan,
        "wheel_speed_rpm": math.nan,
        "density_g_cm3": float(setup_d_row["density_g_cm3"]),
        "ess_mg": float(setup_d_row["effective_sample_mass_estimate_mg"]),
    },
    {
        "setup": "E",
        "scenario": "E1",
        "basis": "Swept capsule unique exposure",
        "probe_diameter_mm": 5.0,
        "penetration_depth_mm": ALIGNED_PENETRATION_DEPTH_MM,
        "throughput_kg_h": ALIGNED_THROUGHPUT_KG_H,
        "acquisition_time_s": ALIGNED_ACQUISITION_TIME_S,
        "spoon_interval_s": math.nan,
        "wheel_speed_rpm": math.nan,
        "density_g_cm3": ALIGNED_DENSITY_G_CM3,
        "ess_mg": setup_e_5mm["effective_sample_size_mg"],
    },
    {
        "setup": "E",
        "scenario": "E2",
        "basis": "Swept capsule unique exposure",
        "probe_diameter_mm": 25.0,
        "penetration_depth_mm": ALIGNED_PENETRATION_DEPTH_MM,
        "throughput_kg_h": ALIGNED_THROUGHPUT_KG_H,
        "acquisition_time_s": ALIGNED_ACQUISITION_TIME_S,
        "spoon_interval_s": math.nan,
        "wheel_speed_rpm": math.nan,
        "density_g_cm3": ALIGNED_DENSITY_G_CM3,
        "ess_mg": setup_e_25mm["effective_sample_size_mg"],
    },
]

comparison_df = pd.DataFrame(comparison_rows)
scenario_order = {"A1": 1, "B2": 2, "C1": 3, "D1": 4, "E1": 5, "E2": 6}
comparison_df["scenario_order"] = comparison_df["scenario"].map(scenario_order)
comparison_df["rank"] = comparison_df["ess_mg"].rank(ascending=False, method="dense").astype(int)
comparison_df = comparison_df.sort_values("scenario_order").reset_index(drop=True)

comparison_table_df = comparison_df.rename(
    columns={
        "setup": "Setup",
        "scenario": "Scenario",
        "basis": "Basis",
        "probe_diameter_mm": "Probe mm",
        "penetration_depth_mm": "Depth mm",
        "throughput_kg_h": "Throughput kg/h",
        "acquisition_time_s": "Acq s",
        "spoon_interval_s": "Spoon s",
        "wheel_speed_rpm": "RPM",
        "density_g_cm3": "Density g/cm^3",
        "ess_mg": "ESS mg",
        "rank": "Rank",
    }
)[[
    "Setup",
    "Scenario",
    "Basis",
    "Probe mm",
    "Depth mm",
    "Throughput kg/h",
    "Acq s",
    "Spoon s",
    "RPM",
    "Density g/cm^3",
    "ESS mg",
    "Rank",
]]

for column in ["Probe mm", "Depth mm", "Throughput kg/h", "Acq s", "Spoon s", "RPM", "Density g/cm^3", "ESS mg"]:
    comparison_table_df[column] = comparison_table_df[column].map(lambda value: round(value, 3) if pd.notna(value) else value)

comparison_table_df = comparison_table_df.fillna("")

## Main Comparison Table

The table below is the primary notebook output.

In [5]:
comparison_table_df

,Setup,Scenario,Basis,Probe mm,Depth mm,Throughput kg/h,Acq s,Spoon s,RPM,Density g/cm^3,ESS mg,Rank
0,A,A1,Static illuminated volume,5.000,1.000,,1.800,,,0.400,7.854,6
1,B,B2,Static mass times spoon presentations,5.000,1.000,95.000,3.600,3.000,,0.400,9.425,5
2,C,C1,Blocked deterministic renewal,5.000,1.000,95.000,1.800,,32.000,0.400,120.281,2
3,D,D1,Flow-refresh proxy,5.000,1.000,95.000,1.800,,,0.400,56.213,3
4,E,E1,Swept capsule unique exposure,5.000,1.000,95.000,1.800,,,0.400,38.093,4
5,E,E2,Swept capsule unique exposure,25.000,1.000,95.000,1.800,,,0.400,347.547,1


## Short Interpretation

- Setup E with the 25 mm spot is highest because the large instantaneous footprint dominates the total unique-exposure mass.
- Setup A is lowest because the model is only a static illuminated volume with no renewal term.
- Setup C is above Setup D because 32 RPM generates more refresh events in 1.8 s than the pipe-flow case at 95 kg/h, even after blocking is removed.
- Setup E drops sharply when the spot shrinks from 25 mm to 5 mm because both the circular footprint and the swept increment contract.
- Setup B at 3.6 s only slightly exceeds Setup A because 3 s spoon flips yield just 1.2 spoon presentations during one spectrum.

## Export and Reporting

This section writes a timestamped comparison folder under `reports/setup_comparison/`, saves the aligned scenario table and the final comparison table as CSV, writes metadata as JSON, and generates a Word document with the notebook markdown rendered together with the comparison table.

In [7]:
from docx import Document

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.reporting_utils as reporting_utils

reporting_utils = importlib.reload(reporting_utils)
create_run_directory = reporting_utils.create_run_directory
save_dataframe_csv = reporting_utils.save_dataframe_csv
write_metadata_json = reporting_utils.write_metadata_json
extract_markdown_cells_from_notebook = reporting_utils.extract_markdown_cells_from_notebook
add_markdown_cells_to_docx = reporting_utils.add_markdown_cells_to_docx


def add_dataframe_to_docx(document: Document, df: pd.DataFrame, heading: str) -> None:
    document.add_heading(heading, level=1)
    table = document.add_table(rows=1, cols=len(df.columns))
    table.style = "Table Grid"
    for cell, column in zip(table.rows[0].cells, df.columns):
        cell.text = str(column)
    for row in df.itertuples(index=False):
        cells = table.add_row().cells
        for index, value in enumerate(row):
            if pd.isna(value):
                cells[index].text = ""
            elif isinstance(value, float):
                cells[index].text = f"{value:,.3f}"
            else:
                cells[index].text = str(value)


timestamp, run_dir = create_run_directory(PROJECT_ROOT, "setup_comparison")
comparison_csv_path = save_dataframe_csv(comparison_df.drop(columns=["scenario_order"]), run_dir / "comparison_table.csv")
scenario_csv_path = save_dataframe_csv(scenario_df, run_dir / "comparison_inputs.csv")

metadata = {
    "setup_name": "setup_comparison",
    "aligned_density_g_cm3": ALIGNED_DENSITY_G_CM3,
    "aligned_penetration_depth_mm": ALIGNED_PENETRATION_DEPTH_MM,
    "aligned_throughput_kg_h": ALIGNED_THROUGHPUT_KG_H,
    "default_acquisition_time_s": ALIGNED_ACQUISITION_TIME_S,
    "note": "Setup B 1.8 s scenario intentionally omitted; Setup A acquisition time is reported for alignment only.",
    "source_report_runs": {
        "setup_a": str(latest_run_dir("setup_a")),
        "setup_b": str(latest_run_dir("setup_b")),
        "setup_c": str(latest_run_dir("setup_c")),
        "setup_d": str(latest_run_dir("setup_d")),
        "setup_e": str(latest_run_dir("setup_e")),
    },
}
write_metadata_json(metadata, run_dir / "metadata.json")

document = Document()
document.add_heading("ESS Comparison Across Setups A-E", level=0)
markdown_cells = extract_markdown_cells_from_notebook(NOTEBOOK_PATH)
add_markdown_cells_to_docx(document, markdown_cells, run_dir)
add_dataframe_to_docx(document, comparison_table_df, "Comparison Table")
word_report_path = run_dir / "setup_comparison_report.docx"
document.save(word_report_path)

print(f"Report run directory: {run_dir}")
print(f"Comparison CSV: {comparison_csv_path}")
print(f"Scenario CSV: {scenario_csv_path}")
print(f"Word report: {word_report_path}")

Report run directory: C:\temp files\EFS\reports\setup_comparison\20260424_081826
Comparison CSV: C:\temp files\EFS\reports\setup_comparison\20260424_081826\comparison_table.csv
Scenario CSV: C:\temp files\EFS\reports\setup_comparison\20260424_081826\comparison_inputs.csv
Word report: C:\temp files\EFS\reports\setup_comparison\20260424_081826\setup_comparison_report.docx
